# MWBE Certification Data Ingestion & Feature Engineering

## Project Overview

This project analyzes publicly available NYC MWBE certification data to explore:
- certification patterns
- vendor operational readiness
- business maturity
- procurement participation
- data quality and completeness
- vendor intelligence opportunities

The project combines:
- exploratory data analysis (EDA)
- operational analytics
- feature engineering
- procurement enrichment
- vendor intelligence workflows

The broader objective is to demonstrate how analytics and feature engineering can complement modern certification and vendor management systems.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

# Data Source

The primary dataset used in this project was the:

## NYC SBS Certified Business List

Source:
- NYC Department of Small Business Services (SBS)
- NYC Open Data

# Initial Data Profiling

Initial profiling steps included:
- schema inspection
- shape validation
- datatype review
- missing value analysis
- operational field assessment

The dataset contained:
- 11,500+ vendor records
- 50+ operational and business profile attributes

The profiling process identified:
- malformed date fields
- inconsistent formatting
- sparse operational capacity fields
- missing geospatial attributes
- optional enrichment fields


In [ ]:
mwbe_df = pd.read_csv(
    "../data/raw/sbs_certified_businesses.csv"
)

mwbe_df.head()

In [ ]:
mwbe_df.info()

In [ ]:
mwbe_df.isnull().sum()

In [ ]:
mwbe_df.shape

# Data Cleaning & Standardization

## Column Standardization

Column names were standardized by:
- converting to lowercase
- replacing spaces with underscores
- removing inconsistent formatting

This improved:
- transformation consistency
- SQL compatibility
- downstream pipeline reliability

In [ ]:
df_raw = mwbe_df.copy()
df_clean = mwbe_df.copy()

In [ ]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

In [ ]:
df_clean["vendor_name_clean"] = (
    df_clean["vendor_formal_name"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(",", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(" LLC", "", regex=False)
    .str.replace(" INC", "", regex=False)
    .str.replace(" CORP", "", regex=False)
    .str.replace(" CORPORATION", "", regex=False)
    .str.replace(" LTD", "", regex=False)
)

In [ ]:
df_clean.columns

In [ ]:
df_clean.columns.tolist()

## Certification Filtering

The analysis focused primarily on:
- MBE-certified firms
- WBE-certified firms
- MWBE-certified firms

Businesses classified solely as:
- LBE (Locally Based Enterprise)
- EBE (Emerging Business Enterprise)

were excluded unless they also possessed MWBE-related certifications.

This ensured the project remained focused on supplier diversity and certification analytics.


In [ ]:
cert_col = "certification"  

df_clean = df_clean[
    df_clean[cert_col]
    .astype(str)
    .str.contains("MBE|WBE|MWBE", case=False, na=False)
]

In [ ]:
df_clean[cert_col].value_counts()

## Renewal Date Cleaning

The certification renewal date field contained malformed records with:
- duplicated values
- truncated date strings
- delimiter inconsistencies

To standardize the field:
- the first valid date token was extracted
- malformed suffixes were removed
- values were converted into datetime format

Invalid records were safely coerced to null values to preserve pipeline stability.

In [ ]:
date_col = "certification_renewal_date" 

df_clean["certification_renewal_date"] = (
    df_clean[date_col]
    .astype(str)
    .str.split(";")
    .str[0]
)

df_clean["certification_renewal_date"] = pd.to_datetime(
    df_clean["certification_renewal_date"],
    errors="coerce"
)

In [ ]:
df_clean["certification_renewal_date"].head(10)

In [ ]:
df_clean["certification_renewal_date"].dtype

# Feature Engineering

To support operational analytics and vendor intelligence workflows, several engineered features were created.

## Certification Lifecycle Features

### renewal_year
Extracted certification renewal year from renewal dates.

### months_until_renewal
Calculated the number of months remaining until certification renewal.

### renewal_urgency
Segmented businesses into operational renewal categories:
- Expired/Past Due
- Due Within 3 Months
- Due Within 12 Months
- Long-Term Active

### missing_renewal_date
Binary indicator identifying missing renewal dates.



In [ ]:
df_clean["missing_renewal_date"] = (
    df_clean["certification_renewal_date"]
    .isnull()
    .astype(int)
)

In [ ]:
df_clean["renewal_year"] = (
    df_clean["certification_renewal_date"]
    .dt.year
)

today = pd.Timestamp.today()

df_clean["months_until_renewal"] = (
    (
        df_clean["certification_renewal_date"]
        - today
    ).dt.days / 30
).round(1)

In [ ]:
df_clean["renewal_urgency"] = pd.cut(
    df_clean["months_until_renewal"],
    bins=[-9999, 0, 3, 12, 9999],
    labels=[
        "Expired/Past Due",
        "Due Within 3 Months",
        "Due Within 12 Months",
        "Long-Term Active"
    ]
)

In [ ]:
df_clean[[
    "certification_renewal_date",
    "renewal_year",
    "months_until_renewal",
    "renewal_urgency"
]].head()

# Data Quality & Missingness Analysis

The dataset exhibited varying levels of completeness across operational fields.

Core business identity fields were highly complete, while advanced operational capacity fields such as:
- bonding limits
- project experience
- construction specialization

contained substantial missingness.

This missingness was treated as:
- an operational signal
- a data quality indicator
- a potential proxy for vendor maturity and onboarding completeness

rather than simply discarded data.

In [ ]:
missing_summary = (
    df_clean.isnull()
    .mean()
    .sort_values(ascending=False)
    * 100
).round(2)

missing_summary.head(15)

In [ ]:


missing_summary.head(15).plot(
    kind="barh"
)

plt.title("Top Missing Fields (%)")
plt.xlabel("Percent Missing")

plt.show()

In [ ]:
df_clean.head ()

## Data Quality Features

### profile_completeness_score
Calculated the number of populated critical profile fields.

### profile_completeness_pct
Percentage-based completeness metric.

These features supported:
- operational readiness analysis
- vendor profile quality scoring
- segmentation workflows


In [ ]:
important_cols = [
    "vendor_formal_name",
    "certification",
    "business_description",
    "website",
    "date_of_establishment",
    "certification_renewal_date",
    "naics_sector",
    "naics_title",
    "borough"
]

In [ ]:
df_clean["has_website"] = (
    df_clean["website"]
    .notnull()
    .astype(int)
)

In [ ]:
df_clean["profile_completeness_score"] = (
    df_clean[important_cols]
    .notnull()
    .sum(axis=1)
)

df_clean["profile_completeness_pct"] = (
    df_clean["profile_completeness_score"]
    / len(important_cols)
).round(2)

## Business Maturity Features

### years_in_business
Calculated vendor age using establishment date.

### project_experience_count
Counted the number of submitted project experience records.

### contract_capacity_segment
Segmented vendors by largest reported contract value:
- Small Capacity
- Medium Capacity
- Large Capacity
- Enterprise Capacity

### vendor_capacity_score
Composite operational capacity metric derived from:
- largest contract value
- bonding limits
- project experience

In [ ]:
df_clean["date_of_establishment"] = pd.to_datetime(
    df_clean["date_of_establishment"],
    errors="coerce"
)

In [ ]:
current_year = pd.Timestamp.today().year

df_clean["years_in_business"] = (
    current_year
    - df_clean["date_of_establishment"].dt.year
)

In [ ]:
df_clean["largest_value_of_contract"] = (
    df_clean["largest_value_of_contract"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

df_clean["largest_value_of_contract"] = pd.to_numeric(
    df_clean["largest_value_of_contract"],
    errors="coerce"
)

In [ ]:
df_clean["largest_contract_rank"] = (
    df_clean["largest_value_of_contract"]
    .rank(pct=True)
)

df_clean["bonding_rank"] = (
    df_clean["aggregate_bonding_limit"]
    .rank(pct=True)
)

df_clean["experience_rank"] = (
    df_clean["project_experience_count"]
    .rank(pct=True)
)

df_clean["vendor_capacity_score"] = (
    df_clean[
        [
            "largest_contract_rank",
            "bonding_rank",
            "experience_rank"
        ]
    ]
    .mean(axis=1)
    .round(2)
)

In [ ]:
df_clean["contract_capacity_segment"] = pd.cut(
    df_clean["largest_value_of_contract"],
    bins=[
        0,
        100000,
        1000000,
        10000000,
        100000000
    ],
    labels=[
        "Small",
        "Medium",
        "Large",
        "Enterprise"
    ]
)

In [ ]:
df_clean["contract_capacity_segment"] = pd.cut(
    df_clean["largest_value_of_contract"],
    bins=[0, 100000, 1000000, 10000000, np.inf],
    labels=[
        "Small Capacity",
        "Medium Capacity",
        "Large Capacity",
        "Enterprise Capacity"
    ]
)

In [ ]:
df_clean.to_csv(
    "../data/processed/mwbe_vendor_intelligence.csv",
    index=False
)

In [ ]:
df_clean["renewal_urgency"].value_counts()

In [ ]:
urgency_counts = (
    df_clean["renewal_urgency"]
    .value_counts()
)

urgency_counts.plot(kind="bar")

plt.title(
    "Certification Renewal Urgency Distribution"
)

plt.xlabel("Renewal Status")
plt.ylabel("Business Count")

plt.xticks(rotation=15)

plt.show()

In [ ]:
money_cols = [
    "largest_value_of_contract",
    "aggregate_bonding_limit",
    "value_of_contract_job_exp_2",
    "value_of_contract_job_exp_3"
]

for col in money_cols:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

In [ ]:
df_clean["date_of_establishment"] = pd.to_datetime(
    df_clean["date_of_establishment"],
    errors="coerce"
)

current_year = pd.Timestamp.today().year

df_clean["years_in_business"] = (
    current_year - df_clean["date_of_establishment"].dt.year
)

## Digital Readiness Features

### has_website
Binary indicator for website presence.

### passport_enrolled_flag
Binary indicator identifying PASSPort enrollment.

### digital_readiness_score
Composite metric combining:
- website presence
- PASSPort enrollment
- profile completeness

In [ ]:
df_clean["has_website"] = (
    df_clean["website"]
    .notnull()
    .astype(int)
)

df_clean["passport_enrolled_flag"] = (
    df_clean["enrolled_in_passport"]
    .astype(str)
    .str.upper()
    .str.contains("YES|Y|TRUE", na=False)
    .astype(int)
)

In [ ]:
df_clean["digital_readiness_score"] = (
    (
        df_clean["has_website"]
        + df_clean["passport_enrolled_flag"]
        + df_clean["profile_completeness_pct"]
    ) / 3
).round(2)

In [ ]:

experience_cols = [
    "name_of_client_job_exp_1",
    "name_of_client_job_exp_2",
    "name_of_client_job_exp_3"
]

df_clean["project_experience_count"] = (
    df_clean[experience_cols]
    .notnull()
    .sum(axis=1)
)

## Refinement of Profile Completeness Scoring

The initial profile completeness score was created using a smaller subset of core vendor identity and certification fields.

As the exploratory analysis progressed and additional operational attributes were evaluated, the completeness framework was expanded to include a broader set of fields


In [ ]:
important_cols = [
    "vendor_formal_name",
    "business_description",
    "certification",
    "certification_renewal_date",
    "ethnicity",
    "city",
    "state",
    "postcode",
    "website",
    "date_of_establishment",
    "naics_sector",
    "naics_subsector",
    "naics_title",
    "nigp_codes",
    "enrolled_in_passport"
]

df_clean["profile_completeness_score"] = (
    df_clean[important_cols]
    .notnull()
    .sum(axis=1)
)

df_clean["profile_completeness_pct"] = (
    df_clean["profile_completeness_score"] / len(important_cols)
).round(2)

In [ ]:
df_clean["certification_count"] = (
    df_clean["certification"]
    .astype(str)
    .str.split(",|;|/")
    .apply(len)
)

df_clean["multi_certified_flag"] = (
    df_clean["certification_count"] > 1
).astype(int)

## Industry & Geographic Feature Engineering

To support industry segmentation and geospatial analytics, several additional standardized features were engineered from the raw vendor profile data.

These features improved:
- grouping consistency
- dashboard readiness
- geographic analysis
- operational segmentation
- downstream analytical workflows

In [ ]:
df_clean["industry_specialization"] = (
    df_clean["naics_sector"]
    .astype(str)
    .str.strip()
)

df_clean["naics_title_clean"] = (
    df_clean["naics_title"]
    .astype(str)
    .str.upper()
    .str.strip()
)

In [ ]:
df_clean["has_geocode"] = (
    df_clean["latitude"].notnull()
    & df_clean["longitude"].notnull()
).astype(int)

df_clean["borough_clean"] = (
    df_clean["borough"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df_clean.loc[
    df_clean["borough_clean"].isin(["NAN", "NONE", ""]),
    "borough_clean"
] = np.nan

## Operational Readiness Features

### operational_readiness_score
Composite operational intelligence metric derived from:
- profile completeness
- digital readiness
- vendor capacity

### operational_readiness_segment
Segmented vendors into:
- Low Readiness
- Moderate Readiness
- High Readiness

These features supported:
- vendor prioritization
- opportunity gap analysis
- operational intelligence workflows

In [ ]:
score_cols = [
    "profile_completeness_pct",
    "digital_readiness_score",
    "vendor_capacity_score"
]

df_clean["operational_readiness_score"] = (
    df_clean[score_cols]
    .mean(axis=1)
    .round(2))

In [ ]:
df_clean["operational_readiness_segment"] = pd.cut(
    df_clean["operational_readiness_score"],
    bins=[0, 0.33, 0.66, 1.0],
    labels=[
        "Low Readiness",
        "Moderate Readiness",
        "High Readiness"
    ],
    include_lowest=True
)

In [ ]:
feature_cols = [
    "vendor_formal_name",
    "certification",
    "years_in_business",
    "months_until_renewal",
    "renewal_urgency",
    "profile_completeness_pct",
    "project_experience_count",
    "contract_capacity_segment",
    "vendor_capacity_score",
    "digital_readiness_score",
    "operational_readiness_score",
    "operational_readiness_segment"
]

df_clean[feature_cols].head()

In [ ]:
df_clean.to_csv(
    "../data/processed/mwbe_vendor_intelligence.csv",
    index=False
)